# Vector index and database

Let's look at a simple example of a vector store with numpy.

In [ ]:
import numpy as np

Imagine this was the result of your document ingestion and chunking:

In [ ]:
docs = [
    {"id": 0, 
     "text": "Refunds are available within 30 days of purchase.", 
     "metadata": {"source": "policy.pdf"}},
    {"id": 1, 
     "text": "Support is available 24/7 via chat.", 
     "metadata": {"source": "support.md"}},
    {"id": 2, 
     "text": "Annual plans renew automatically.", 
     "metadata": {"source": "billing.md"}},
]

And imagine this was the result of obtaining embeddings from an embedding model:

In [ ]:
embeddings = np.array([
    [0.1, 0.2, 0.3],
    [0.0, 0.1, 0.9],
    [0.5, 0.2, 0.1],
])

We also need an algorithm for similarity search.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) #/ (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
# this iterates over our document embeddings
# and uses cosine similarity to assess similarity of "query_emb" 
# with our documents

def naive_similarity_search(query_emb, k = 2):
    scores = []
    for i, emb in enumerate(embeddings):
        score = cosine_similarity(query_emb, emb)
        scores.append((score, docs[i]))
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:k]

Let's say we have a query that gets embedded as `[0.09, 0.18, 0.25]`.  We can then use the similarity search to return most similar chunks.

In [ ]:
query_embedding = np.array([0.09, 0.18, 0.25])  

In [ ]:
top_matches = naive_similarity_search(query_embedding, k=2)

In [ ]:
for score, doc in top_matches:
    print(f"{score:.3f} :: {doc['text']} (from {doc['metadata']['source']})")

That’s all a vector index is doing: store vectors + compute nearest neighbors. Libraries just make it fast, scalable, and convenient.

# Build index with FAISS

FAISS (Facebook AI Similarity Search):

* Library for fast similarity search over large collections of vectors.
* Has a Python API on top of optimized C++/GPU code.
* Typically stores only vectors; you store text/metadata in parallel structures.

In [ ]:
import faiss

In [ ]:
# Suppose you already have chunked_docs from earlier:
# chunked_docs = [
#   {"id": 0, "text": "...", "metadata": {...}},
#   {"id": 1, "text": "...", "metadata": {...}},
#   ...
# ]

chunked_docs = docs

We will explicitly generate embeddings now for our chunked docs:

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Compute embeddings for all chunks
texts = [doc["text"] for doc in chunked_docs]
embs = model.encode(texts, convert_to_numpy=True)

In [ ]:
embs.shape

In [ ]:
embs

FAISS expects float32, which we already have, but just in case...

In [ ]:
embs = embs.astype("float32")

To use inner product (cosine-like similarity), we can use `faiss.IndexFlatIP`, a flat index designed for exact nearest neighbor search using inner product (dot product) similarity.

In [ ]:
dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)

To use cosine similarity via inner product, we can further use `faiss.normalize_L2` to normalize input vectors to unit length using the L2 (Euclidean) norm.  This means each vector is divided by its L2 norm, resulting in vectors where the sum of the squares of the elements equals 1.

It actually happens that `embs` already has normalized vectors...

In [ ]:
cosine_similarity(embs[0], embs[0])

But anyhow...

In [ ]:
faiss.normalize_L2(embs)

Now add vectors to the vector index:

In [ ]:
index.add(embs)

In [ ]:
index.ntotal

We'll keep a parallel list of docs for lookup

In [ ]:
doc_store = chunked_docs

In [ ]:
doc_store

And now, how do we use our index to find similar chunks as our query?

In [ ]:
test = "What's our refund policy?"
test_emb = model.encode([test], convert_to_numpy=True).astype("float32")
faiss.normalize_L2(test_emb)
index.search(test_emb, 2)

These are distances (aka similarity scores) and the matching indices of the indexed embeddings.

In [ ]:
doc_store[0]

In [ ]:
def faiss_search(query, k = 3):
    
    # Embed query
    q_emb = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)

    # Search top-k
    # By default, index.search returns sorted results (in descending order of similarity)
    distances, indices = index.search(q_emb, k)  # shapes: (1, k)
    indices = indices[0]
    distances = distances[0]

    results = []
    for idx, score in zip(indices, distances):
        doc = doc_store[idx]
        results.append({
            "score": float(score),
            "text": doc["text"],
            "metadata": doc["metadata"],
        })
    return results

In [ ]:
hits = faiss_search("What's our refund policy?", k=3)

for h in hits:
    print(f"{h['score']:.3f} :: {h['metadata']['source']} :: {h['text'][:80]}...")

Notes:
* FAISS is just the index: it knows about vectors and integer IDs.
* You store your text + metadata in a separate structure (doc_store here).
* It can scale to millions/billions of vectors using advanced indexes (IVF, HNSW, etc.), but the API idea remains the same.

# Using Chroma (vector database with metadata)

Chroma:

* A Python‑friendly vector DB.
* Stores documents, embeddings, and metadata together.
* Has a simple .add() / .query() API that feels nice in a RAG app.

In [ ]:
import chromadb

Create a client object that can be used to interact with ChromaDB.

In [ ]:
# client = chromadb.Client()  # <-- creates ephemeral client that runs entirely in memory
client = chromadb.PersistentClient(path="chroma") # <-- creates client that stores data to local directory

We use `get_or_create_collection()` instead of `create_collection()`. This method will return the existing collection if it exists, or create it if it doesn’t.

In [ ]:
collection = client.get_or_create_collection(name="docs")

Now we'll collect the information to store for each chunk.

In [ ]:
# Assume chunked_docs as before
ids = [str(doc["id"]) for doc in chunked_docs]
documents = [doc["text"] for doc in chunked_docs]
metadatas = [doc["metadata"] for doc in chunked_docs]

In [ ]:
print('ids:\n', ids)
print('documents:\n', documents)
print('metadatas:\n', metadatas)

We'll also store text embedding vectors in the database.

In [ ]:
embeddings = model.encode(documents).tolist()

Here we add all of the information into the database.

In [ ]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
)

To see what's in our database:

In [ ]:
# Retrieve all documents (with metadata and embeddings)
results = collection.get(include=["documents", "metadatas", "embeddings"])

# Print all documents
for i, doc in enumerate(results["documents"]):
    print(f"ID: {results['ids'][i]}")
    print(f"Document: {doc}")
    print(f"Document-db: {results['documents'][i]}")
    print(f"Metadata: {results['metadatas'][i]}")
    print(f"Embedding[:10]: {results['embeddings'][i][:10]}")
    print("-" * 50)

If we have a query string, we can check it's similarity with other texts in our database.

First we need to encode it (with the same embedding model).

In [ ]:
query_texts=["This is a query document about refunds"]
q_emb = model.encode(query_texts).tolist()

Then we can use the `query` method to return the top-k similar chunks.

In [ ]:
results = collection.query(
    query_embeddings=q_emb,
    n_results=2
)
results

Here the distances are lower-valued for more-similar embeddings.  For our ChromaDB PersistentClient, the distance metric defaults to L2 (Euclidean distance).

In [ ]:
def chroma_search(query, k=5, product=None):
    
    q_emb = model.encode([query]).tolist()

    # optionally also including a filter!
    where = {"product": product} if product is not None else None

    result = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        where=where,  # optional metadata filter
    )

    # result["documents"], result["metadatas"], result["distances"] are lists-of-lists
    docs = result["documents"][0]
    metas = result["metadatas"][0]
    dists = result["distances"][0]

    hits = []
    for text, meta, dist in zip(docs, metas, dists):
        hits.append({
            "score": float(dist),
            "text": text,
            "metadata": meta,
        })
    return hits

hits = chroma_search("What's our refund policy?", k=3, product='Toys')
for h in hits:
    print(h["score"], h["metadata"], h["text"][:80], "...")


Remember:
* Chunking and embeddings give us a searchable representation of our knowledge.
* Vector databases are the tools that make that search fast and scalable.